# RoPE — Cached Rotary Position Embedding Triton Kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qooba/bielik-anatomy-triton/blob/main/notebooks/ep04-rope.ipynb)

Part of the [Bielik Anatomy](https://github.com/qooba/bielik-anatomy-triton) series — Episode 4: RoPE. Full write-up: [ep04-rope.md](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep04-rope.md).

RoPE rotates Q and K vectors so attention becomes position-aware. This notebook runs the single-pass cached Triton RoPE kernel (`kernels/attention/rope_cached.py`), which precomputes the cos/sin cache to remove transcendental ops from the hot path.

This notebook lets you run this kernel's correctness check and benchmark on a free Colab GPU — no local NVIDIA hardware required.

**Runtime:** select a GPU runtime before running: `Runtime -> Change runtime type -> T4 GPU` (free tier works fine).

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = "bielik-anatomy-triton"
if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/qooba/bielik-anatomy-triton {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 1. Correctness check

Compare the Triton kernel output against a plain PyTorch reference implementation.

In [ ]:
import torch
from kernels.attention.rope_cached import build_rope_cache, apply_rope_cached_

torch.manual_seed(0)
batch, seq_len, num_heads, head_dim = 2, 16, 4, 64
x_triton = torch.randn(batch, seq_len, num_heads, head_dim, device="cuda", dtype=torch.float32)
x_ref = x_triton.clone()
positions = torch.arange(seq_len, device="cuda", dtype=torch.int32).unsqueeze(0).expand(batch, -1)

cos_cache, sin_cache = build_rope_cache(seq_len, head_dim, device="cuda")
apply_rope_cached_(x_triton, positions, cos_cache, sin_cache)

def rope_ref(x, cos_cache, sin_cache):
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    cos = cos_cache[: x.shape[1]].view(1, -1, 1, half)
    sin = sin_cache[: x.shape[1]].view(1, -1, 1, half)
    out1 = x1 * cos - x2 * sin
    out2 = x1 * sin + x2 * cos
    return torch.cat([out1, out2], dim=-1)

out_ref = rope_ref(x_ref, cos_cache, sin_cache)

max_diff = (x_triton - out_ref).abs().max().item()
print(f"max abs diff vs PyTorch reference: {max_diff:.6f}")
assert torch.allclose(x_triton, out_ref, atol=1e-4, rtol=1e-4)
print("Correctness check passed.")

## 2. Benchmark

Runs the same sweep as `make benchmark-rope` and regenerates the plots used in the episode doc. Takes roughly 1-3 minutes on a T4.

In [ ]:
!PYTHONPATH=. python benchmarks/attention/benchmark_rope_cached.py --save-plots --plot-dir=docs/plots/attention/

## 3. Results

In [ ]:
from IPython.display import Image, display

plot_dir = "docs/plots/attention/"
for fname in [
    "rope-cached-bandwidth-vs-heads.png",
    "rope-cached-bandwidth-vs-seq-len.png",
]:
    display(Image(filename=f"{plot_dir}{fname}"))

---
[Full episode write-up](https://github.com/qooba/bielik-anatomy-triton/blob/main/docs/ep04-rope.md) · [Series overview](https://github.com/qooba/bielik-anatomy-triton/blob/main/README.md)